---
#LangChain을 활용한 AI 자동화


---

# 학습내용
- 환경구축, 필수 라이브러리 설치
- ZeroShotPromptTemplate, FewShotPromptTemplate
- Callbacks - streaming, ExampleSelector
- MessagePromptTemplate, FewShotChatMessagePromptTemplate
- Partial Prompt
- 여러 개의 프롬프트 연결하기, Multi Chain, 사용 토큰량 확인





# 환경 구축


- 구글 드라이브 연동

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/Colab Notebooks/인사교1_LangChain_20260624

/content/drive/MyDrive/Colab Notebooks/인사교1_LangChain_20260624


## 필수 라이브러리 설치

- LangChain 설치 : LangChain 1.0을 설치하면 langchain-core 등 필수 라이브러리가 함께 설치

In [3]:
!pip install -qU langchain>=1.0.0 langchain_community

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


- 모델별 패키지 설치

In [4]:
!pip install -qU langchain-openai langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.3 MB/s eta 0:00:00


- 벡터 DB 설치

In [5]:
# langchain-chroma, faiss-cpu : 벡터 DB를 오픈소스로 제공
!pip install -qU langchain-chroma faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

- 라이브러리 가져오기

In [6]:
import os

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
# 다양한 프로바이더 모델들을 초기화해주는 기능
from langchain.chat_models import init_chat_model

from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import FewShotPromptTemplate
from langchain_core.prompts import FewShotChatMessagePromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

# Runnable 인스턴스 생성 (체인 연결이 가능한 형태로 변환)
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel
from langchain_core.runnables import RunnableLambda

from langchain_core.output_parsers import StrOutputParser

# stream 기능
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
# 유사도가 높은 예시를 선택
from langchain_core.example_selectors import SemanticSimilarityExampleSelector

# 벡터 DB
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import Chroma

# 사용한 토큰량, 요금을 반환하는 함수
from langchain_community.callbacks import get_openai_callback

# 마크다운 형태로 출력
from IPython.display import display, Markdown

# 구조화된 출력을 위한 기능
from pydantic import BaseModel, Field

/tmp/ipykernel_5490/2029655133.py:27: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


- API 키 등록

In [7]:
# 파일에서 API 키 읽기
with open("./key/openai_key.txt", "r") as f:
    api_key = f.read().strip()

os.environ['OPENAI_API_KEY'] = api_key

In [8]:
# 파일에서 API 키 읽기
with open("./key/google_key.txt", "r") as f:
    api_key = f.read().strip()

os.environ['GOOGLE_API_KEY'] = api_key

# Zero-Shot Prompting
  - AI 모델이 사전 학습 없이 새로운 작업이나 상황에 대응하는 방식
  - 즉, 모델이 해당 작업에 대한 구체적인 예시나 학습 데이터를 제공받지 않고도 바로 응답하거나 작업을 수행할 수 있는 능력
  - 예를 들어, AI에게 특정 질문을 던졌을 때, 해당 질문에 대한 학습 경험이 전혀 없더라도 일반적인 지식과 언어 이해 능력을 바탕으로 적절한 답변을 제공할 수 있도록 하는 것
  - 항상 완벽하진 않으므로, 프롬프트를 최적화하는 과정이 필요
  - 더 큰 모델이나 최신 모델일수록 일반적인 작업에 대해 더 나은 성능을 나타냄

- Zero-Shot Prompting의 사용 방법
  - 사전 학습된 대규모 언어 모델 준비
    - GPT, BERT와 같은 대규모 사전 학습된 언어 모델을 사용
  - 프롬프트(질문 또는 요청) 작성
    - 구체적인 예시를 제공하지 않고 프롬프트를 구성
  - 모델에 프롬프트 입력
  - 응답 처리 및 활용

- Zero-Shot Prompting의 용도
  - 텍스트 생성을 할 때 효율성과 속도가 중요한 경우
  - 특정 작업에 대한 데이터를 획득하는 데 자원이 부족하거나 어려움을 겪을 때
  - 다양한 프롬프트와 작업을 처리하기 위해 유연성과 적응력이 필요한 경우

- Zero-Shot Prompting 개선 방법
  - Few-Shot Prompting
  - 메모리 기능 사용
  - 체인 결합 (여러개의 언어 모델 연결)
  - RAG (검색, 증강, 생성)
  - TAG (테이블(DB), 증강, 생성)
  - Agent  

# FewShotPromptTemplate

- <font color=red>Few-Shot Prompting</font>
  - AI 모델과 효과적으로 소통하기 위해 <font color=red>소수의 예시(하나에서 다섯 사이)를 제공하여 특정 응답을 요구하거나 특정 작업을 가르치는 기술</font>
  - 모델에게 주어진 예시들이 패턴을 보여주거나 특정한 반응 방식을 설명하면 모델은 예시들을 참조로 사용하여 자신의 반응에서 기대되는 것을 이해
  - 단순한 지시만으로는 모델이 의도된 작업을 충분히 이해하기 어려운 복잡하거나 미묘한 작업에 유용
  - 모델 Finetuning이나 대규모 데이터셋이 실행 가능하지 않은 상황에서 LLM의 능력을 유연하고 효율적으로 활용하기 위한 강력한 도구

- Few-Shot Prompting의 용도
  - 특정 작업을 위한 제한된 학습 데이터가 있는 경우
  - 계산 능력이나 데이터 제약으로 인해 자체 모델을 만드는 것이 불가능한 경우
  - 아이디어를 빠르게 테스트하고 실험할 필요가 있는 경우



- 몇 개의 예시를 제공하여 LLM 모델에게 특정 출력을 지시하는 Few-shot Prompting
  - 예시와 같이 입력에 대응하는 반대말을 출력하는 기능 구현

In [ ]:
# (1) 몇 개의 예시를 작성 (Input의 반대말을 Output으로 출력하는 7개 예시)
examples1 = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "sunny", "output": "rainy"},
    {"input": "surprised", "output": "calm"},
    {"input": "dry", "output": "humid"},
    {"input": "hot", "output": "cold"},
    {"input": "satisfied", "output": "dissatisfied"}
]

In [ ]:
llm = init_chat_model("gpt-4o-mini")

# 예시들을 등록하는 프롬프트
prompt = PromptTemplate.from_template("Input: {input}\nOutput: {output}")

# 퓨삿 프롬프트 설정
fewshot_prompt = FewShotPromptTemplate(
    # 사용할 예시
    examples=examples1,
    # 프롬프트
    example_prompt=prompt,
    # 프롬프트의 마지막에 포함되는 내용
    suffix="Input: {input}\nOutput:",
    # 사용자에게 입력받을 변수
    input_variables=["input"]
)

print(fewshot_prompt.format(input="여름"))

Input: happy
Output: sad

Input: tall
Output: short

Input: sunny
Output: rainy

Input: surprised
Output: calm

Input: dry
Output: humid

Input: hot
Output: cold

Input: satisfied
Output: dissatisfied

Input: 여름
Output:


In [ ]:
chain = fewshot_prompt | llm | StrOutputParser()
print(chain.invoke({"input": "여름"}))
print(chain.invoke({"input": "夜"}))
print(chain.invoke({"input": "홍길동"}))

겨울
日
홍길순


[ 실습 ]

In [ ]:
examples2 = [
    {
        "question":"해도는 빨강색을 좋아하고 병관은 파란색을 좋아한다.",
        "answer":'"해도":"빨강색", "병관":"파란색"'
    },
    {
        "question":"길동은 짜장을 좋아하고, 심청은 짬뽕을 좋아한다.",
        "answer":'"길동":"짜장","심청":"짬뽕"'
    },
    {
        "question":"영표는 국어를 좋아하고 세연은 수학을 좋아한다.",
        "answer":'"영표":"국어", "세연":"수학"'
    },
    {
        "question":"명진은 웹을 강의하고 보미는 사물인터넷을 강의한다.",
        "answer":'"명진":"웹","보미":"사물인터넷"'
    },
    {
        "question":"병관는 서울에 살고, 해도는 광주에 산다.",
        "answer":'"병관":"서울", "해도":"광주"'
    }
]

In [ ]:
llm = init_chat_model("gpt-4o-mini")

# 예시들을 등록하는 프롬프트
prompt = PromptTemplate.from_template("Input: {question}\nOutput: {answer}")

# 퓨삿 프롬프트 설정
fewshot_prompt = FewShotPromptTemplate(
    # 사용할 예시
    examples=examples2,
    # 프롬프트
    example_prompt=prompt,
    # 예시 프롬프트의 처음에 포함되는 내용
    prefix="핵심 키워드를 정리해줘",
    # 예시 프롬프트의 마지막에 포함되는 내용
    suffix="Input: {question}\nOutput:",
    # 사용자에게 입력받을 변수
    input_variables=["question"]
)

print(fewshot_prompt.format(question="사과는 빨간색의 과일이고 수박은 녹색의 채소이다"))

핵심 키워드를 정리해줘

Input: 해도는 빨강색을 좋아하고 병관은 파란색을 좋아한다.
Output: "해도":"빨강색", "병관":"파란색"

Input: 길동은 짜장을 좋아하고, 심청은 짬뽕을 좋아한다.
Output: "길동":"짜장","심청":"짬뽕"

Input: 영표는 국어를 좋아하고 세연은 수학을 좋아한다.
Output: "영표":"국어", "세연":"수학"

Input: 명진은 웹을 강의하고 보미는 사물인터넷을 강의한다.
Output: "명진":"웹","보미":"사물인터넷"

Input: 병관는 서울에 살고, 해도는 광주에 산다.
Output: "병관":"서울", "해도":"광주"

Input: 사과는 빨간색의 과일이고 수박은 녹색의 채소이다
Output:


In [ ]:
chain = fewshot_prompt | llm | StrOutputParser()
print(chain.invoke({"question": "길동은 과일을 좋아하고 길순은 채소를 좋아한다"}))

"길동":"과일", "길순":"채소"


# Callbacks - 스트리밍(streaming)
  - 스트리밍 옵션은 <font color=red>질의에 대한 답변을 실시간</font>으로 받을 때 유용
  - <font color=red>streaming=True</font>로 설정하고 스트리밍으로 답변을 받기 위한 <font color=red>StreamingStdOutCallbackHandler()</font>을 콜백으로 지정

    - 오류가 뜨는 경우 : 라이브러리가 langchain.callbacks.streaming_stdout에서 langchain_core.callbacks.streaming_stdout로 변경됨

In [ ]:
llm = init_chat_model("gpt-4o-mini",
                      streaming=True,
                      callbacks=[StreamingStdOutCallbackHandler()])

chain = llm | StrOutputParser()

answer = chain.invoke("RAG에 대해 500자 내외로 설명해줘")

#display(Markdown(answer))

RAG(정보 검색 및 생성, Retrieval-Augmented Generation)는 인공지능 모델이 정보를 검색하고 이를 기반으로 텍스트를 생성하는 방법론입니다. 전통적인 언어 모델이 사전 훈련된 데이터에 의존하여 응답을 생성하는 반면, RAG는 외부 데이터베이스나 문서에서 관련 정보를 실시간으로 검색하여 더욱 정확하고 정보가 풍부한 답변을 제공합니다.

RAG 프로세스는 크게 두 단계로 나뉩니다. 첫 번째는 정보를 검색하는 단계로, 질문이나 입력에 대해 관련 문서나 데이터를 찾습니다. 두 번째는 이 검색된 정보를 바탕으로 자연어로 응답을 생성하는 단계입니다. 이 과정에서 RAG는 검색된 정보의 맥락을 이해하고, 이를 바탕으로 논리적이고 일관된 결과물을 만들어냅니다.

RAG의 장점은 최신 정보에 접근할 수 있어 더욱 신뢰할 수 있는 답변을 생성하며, 패러다임을 전환하는 데에도 효과적이라는 점입니다. 따라서 RAG는 고객 서비스, 질의응답 시스템, 교육, 연구 등 다양한 분야에서 활용되고 있습니다.

[ 실습 ]

In [ ]:
examples3 = [
    {
        "question": "스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인
"""},
    {
        "question": "네이버의 창립자는 언제 태어났나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일
"""},
    {
        "question": "율곡 이이의 어머니가 태어난 해의 통치하던 왕은 누구인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 율곡 이이의 어머니는 누구인가요?
중간 답변: 율곡 이이의 어머니는 신사임당입니다.
추가 질문: 신사임당은 언제 태어났나요?
중간 답변: 신사임당은 1504년에 태어났습니다.
추가 질문: 1504년에 조선을 통치한 왕은 누구인가요?
중간 답변: 1504년에 조선을 통치한 왕은 연산군입니다.
최종 답변은: 연산군
"""},
    {
        "question": "올드보이와 기생충의 감독이 같은 나라 출신인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 올드보이의 감독은 누구인가요?
중간 답변: 올드보이의 감독은 박찬욱입니다.
추가 질문: 박찬욱은 어느 나라 출신인가요?
중간 답변: 박찬욱은 대한민국 출신입니다.
추가 질문: 기생충의 감독은 누구인가요?
중간 답변: 기생충의 감독은 봉준호입니다.
추가 질문: 봉준호는 어느 나라 출신인가요?
중간 답변: 봉준호는 대한민국 출신입니다.
최종 답변은: 예
"""}
]

# ExampleSelector
- 예시가 많으면 토큰 사용량이 많아지고 따라서 API 사용료가 많이 나옴
- 최종 질문과 유사한 예시 한 두 개만 프롬프트에 포함할 예제를 선택하는 경우에 사용

- 종류
  - <font color=red>BaseExampleSelector</font>: 상속 받아 사용자 정의 ExampleSelector를 생성
  - <font color=red>LengthBasedExampleSelector</font>: 지정 길이가 넘어가지 않도록 예시 개수를 조절
  - <font color=red>MaxMarginalRelevanceExampleSelector</font>: Maximal Marginal Relevance (MMR)을 이용해 질문과 가까우면서도 다양한 예시를 선택
  - <font color=red>SemanticSimilarityExampleSelector</font>: 벡터 임베딩의 코사인 유사도를 이용해 질문과 의미가 가까운 예시를 선택
  - <font color=red>NGramOverlapExampleSelector</font>: n-gram overlap score를 이용해 질문과 가까운 예시를 선택

- 사용자 입력과의 유사도에 따라 예시를 선택해서 LLM 모델에 Few-show Prompting 하기 (질문에 대해 예시와 같이 답변하도록 하는 기능 구현)
  - 예시와 사용자 입력을 Text Embeddings 변환하여, Cosine Similarity가 가장 높은 k개의 예시를 선별해서 Few-shot Prompt에 넣어서 LLM 모델에 전달하는 방법

In [ ]:
llm = init_chat_model(model = "gpt-4o-mini")

# OpenAIEmbeddings() : 사용자 질문과 예시를 임베딩할 모델
# FAISS : 사용할 벡터 DB
# k=2 : 사용할 유사 예시의 갯수
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples1, OpenAIEmbeddings(), FAISS, k=2
)

prompt = PromptTemplate.from_template("Input: {input}\nOutput: {output}")

fewshot_prompt = FewShotPromptTemplate(
    # 전체 예시 대신에 선택한 예시만 등록
    example_selector=example_selector,
    example_prompt=prompt,
    suffix="Input: {input}\nOutput:",
    input_variables=["input"]
)

print(fewshot_prompt.format(input="여름"))

Input: hot
Output: cold

Input: sunny
Output: rainy

Input: 여름
Output:


In [ ]:
chain = fewshot_prompt | llm | StrOutputParser()

print(chain.invoke({"input": "여름"}))
print(chain.invoke({"input": "눈"}))

겨울
비


[ 실습 1 ]
- 질문 : "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"
- 사용할 예시 : examples3
- 선택할 예시 갯수 : 1개

In [ ]:
llm = init_chat_model(model = "gpt-4o-mini")

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples3, OpenAIEmbeddings(), FAISS, k=1
)

prompt = PromptTemplate.from_template("Input: {question}\nOutput: {answer}")

fewshot_prompt = FewShotPromptTemplate(
    # 전체 예시 대신에 선택한 예시만 등록
    example_selector=example_selector,
    example_prompt=prompt,
    suffix="Input: {question}\nOutput:",
    input_variables=["question"]
)

chain = fewshot_prompt | llm | StrOutputParser()

print(chain.invoke({"question": "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"}))

이 질문에 추가 질문이 필요한가요: 예.  
추가 질문: Bill Gates는 언제 태어났나요?  
중간 답변: Bill Gates는 1955년 10월 28일에 태어났습니다.  
추가 질문: Google은 언제 창립되었나요?  
중간 답변: Google은 1998년에 창립되었습니다.  
최종 답변은: 43살. 


# MessagePromptTemplate 활용
- <font color=red>AI 메시지, 시스템 메시지, 사용자 메시지를 사용</font>하여 적절한 맥락을 유지하고, 사용자와의 상호작용을 보다 자연스럽게 처리
  - <font color=red>SystemMessagePromptTemplate</font> : 시스템 프롬프트 설정
  - <font color=red>HumanMessagePromptTemplate</font> : 사용자 프롬프트 설정
- <font color=red>ChatPromptTemplate.from_messages()</font> : 시스템 메시지와 사용자 메시지 템플릿을 포함하는 챗 프롬프트를 구성
- <font color=red>chat_prompt.format_messages()</font> : 사용자의 질문을 포함한 메시지 리스트를 동적으로 생성하여 반환

- 생성된 메시지 리스트는 대화형 인터페이스나 언어 모델과의 상호작용을 위한 입력으로 사용
- 각 메시지는 <font color=red>role</font> (메시지를 말하는 주체, 여기서는 system 또는 user)과 <font color=red>content</font> (메시지의 내용) 속성을 포함
- 이 구조는 시스템과 사용자 간의 대화 흐름을 명확하게 표현하며, 언어 모델이 이를 기반으로 적절한 응답을 생성할 수 있도록 지원

In [10]:
llm = init_chat_model(model="gpt-4o-mini", temperature=0, max_tokens=500)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 천문학과 관련된 지식을 답변하는 전문 에이전트이다"),
        ("human", "{user_input}")
    ]
)

messages = prompt.format_messages(user_input = "태양계에서 가장 큰 행성은 무엇인가요")

messages

[SystemMessage(content='너는 천문학과 관련된 지식을 답변하는 전문 에이전트이다', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='태양계에서 가장 큰 행성은 무엇인가요', additional_kwargs={}, response_metadata={})]

In [11]:
chain = prompt | llm | StrOutputParser()

chain.invoke({"user_input": "태양계에서 가장 큰 행성은 무엇인가요"})

'태양계에서 가장 큰 행성은 목성(Jupiter)입니다. 목성은 지름이 약 139,822킬로미터로, 태양계의 다른 모든 행성보다 훨씬 큽니다. 또한, 목성은 가스 거인으로 분류되며, 두꺼운 대기와 강력한 자기장을 가지고 있습니다.'

# FewShotChatMessagePromptTemplate
- <font color=red>Example Selector를 설정해주고, ChatPromptTemplate.from_messages()에 "human"과 "ai"의 메시지 포맷을 설정</font>
- 입력으로 받는 변수 input_variables=["input"] 을 설정

- Example Selector 의 유사도 검색 문제 해결
  - 유사도 계산시 instruction 과 input 을 사용하고 있지만, instruction 만 사용하여 검색시 제대로된 유사도 결과가 나오지 않음
  - 이를 해결하기 위해 커스텀 유사도 계산을 위한 클래스를 정의하여 사용

In [12]:
examples4 = [
    {
        "instruction": "당신은 회의록 작성 전문가 입니다. 주어진 정보를 바탕으로 회의록을 작성해 주세요",
        "input": "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. 회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. 회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. 팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공했으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다.",
        "answer": """
회의록: XYZ 회사 마케팅 전략 회의
일시: 2023년 12월 25일
장소: XYZ 회사 회의실
참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)

1. 개회
   - 회의는 김수진 팀장의 개회사로 시작됨.
   - 회의의 목적은 2024년 상반기 마케팅 전략 수립 및 새로운 소셜 미디어 캠페인 아이디어 논의.

2. 시장 동향 개요 (김수진)
   - 김수진 팀장은 최근 시장 동향에 대한 분석을 제시.
   - 소비자 행동 변화와 경쟁사 전략에 대한 통찰 공유.

3. 디지털 마케팅 전략 (박지민)
   - 박지민은 디지털 마케팅 전략에 대해 발표.
   - 온라인 광고와 SEO 최적화 방안에 중점을 둠.

4. 소셜 미디어 캠페인 (이준호)
   - 이준호는 새로운 소셜 미디어 캠페인에 대한 아이디어를 제안.
   - 인플루언서 마케팅과 콘텐츠 전략에 대한 계획을 설명함.

5. 종합 논의
   - 팀원들 간의 아이디어 공유 및 토론.
   - 각 전략에 대한 예산 및 자원 배분에 대해 논의.

6. 마무리
   - 다음 회의 날짜 및 시간 확정.
   - 회의록 정리 및 배포는 박지민 담당.
"""
    },
    {
        "instruction": "당신은 요약 전문가 입니다. 다음 주어진 정보를 바탕으로 내용을 요약해 주세요",
        "input": "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량의 보고서입니다. 보고서는 지속 가능한 도시 개발의 중요성, 현재 도시화의 문제점, 그리고 도시 개발을 지속 가능하게 만들기 위한 다양한 전략을 포괄적으로 다루고 있습니다. 이 보고서는 또한 성공적인 지속 가능한 도시 개발 사례를 여러 국가에서 소개하고, 이러한 사례들을 통해 얻은 교훈을 요약하고 있습니다.",
        "answer": """
문서 요약: 지속 가능한 도시 개발을 위한 전략 보고서

- 중요성: 지속 가능한 도시 개발이 필수적인 이유와 그에 따른 사회적, 경제적, 환경적 이익을 강조.
- 현 문제점: 현재의 도시화 과정에서 발생하는 주요 문제점들, 예를 들어 환경 오염, 자원 고갈, 불평등 증가 등을 분석.
- 전략: 지속 가능한 도시 개발을 달성하기 위한 다양한 전략 제시. 이에는 친환경 건축, 대중교통 개선, 에너지 효율성 증대, 지역사회 참여 강화 등이 포함됨.
- 사례 연구: 전 세계 여러 도시의 성공적인 지속 가능한 개발 사례를 소개. 예를 들어, 덴마크의 코펜하겐, 일본의 요코하마 등의 사례를 통해 실현 가능한 전략들을 설명.
- 교훈: 이러한 사례들에서 얻은 주요 교훈을 요약. 강조된 교훈에는 다각적 접근의 중요성, 지역사회와의 협력, 장기적 계획의 필요성 등이 포함됨.

이 보고서는 지속 가능한 도시 개발이 어떻게 현실적이고 효과적인 형태로 이루어질 수 있는지에 대한 심도 있는 분석을 제공합니다.
"""
    }
]

In [13]:
question = {
    "instruction": "회의록을 작성해 주세요",
    "input": "2023년 12월 26일, ABC 기술 회사의 제품 개발 팀은 새로운 모바일 애플리케이션 프로젝트에 대한 주간 진행 상황 회의를 가졌다. 이 회의에는 프로젝트 매니저인 최현수, 주요 개발자인 황지연, UI/UX 디자이너인 김태영이 참석했다. 회의의 주요 목적은 프로젝트의 현재 진행 상황을 검토하고, 다가오는 마일스톤에 대한 계획을 수립하는 것이었다. 각 팀원은 자신의 작업 영역에 대한 업데이트를 제공했고, 팀은 다음 주까지의 목표를 설정했다.",
}

In [15]:
# 모델 생성, 스트리밍 기능 추가
llm = init_chat_model(model="gpt-4o-mini",
                      temperature=0, max_tokens=1000,
                      streaming = True,
                      callbacks = [StreamingStdOutCallbackHandler()])

# (1) 예시 선택기 설정
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples4, OpenAIEmbeddings(), FAISS, k=1
)

# (2) 메세지 프롬트프 생성 (예시 프롬프트)
message = ChatPromptTemplate.from_messages(
    [
        ("system", "{instruction}"),
        ("human", "{input}"),
        ("ai", "{answer}")
    ]
)

# (3) 선택된 예시 1개로 프롬프트를 생성
fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=message
)

# (4) 사용자 질문이 포함된 최종프롬프트 생성
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "{instruction}"),
        fewshot_prompt,
        ("human", "{input}")
    ]
)

# (5) 체인생성
chain = prompt | llm | StrOutputParser()

# (6) 체인 실행
answer = chain.invoke(question)

# (7) 출력
#display(Markdown(answer))

회의록: ABC 기술 회사 제품 개발 팀 주간 진행 상황 회의  
일시: 2023년 12월 26일  
장소: ABC 기술 회사 회의실  
참석자: 최현수 (프로젝트 매니저), 황지연 (주요 개발자), 김태영 (UI/UX 디자이너)  

1. 개회  
   - 최현수 프로젝트 매니저의 개회사로 회의 시작.  
   - 회의의 목적은 새로운 모바일 애플리케이션 프로젝트의 현재 진행 상황 검토 및 다가오는 마일스톤 계획 수립.

2. 진행 상황 업데이트  
   - **황지연 (주요 개발자)**  
     - 현재 개발 진행 상황 보고.  
     - 주요 기능 구현 완료 및 버그 수정 작업 진행 중.  
   - **김태영 (UI/UX 디자이너)**  
     - 사용자 인터페이스 디자인 최종 검토 완료.  
     - 사용자 피드백 반영을 위한 수정 사항 제안.

3. 마일스톤 계획  
   - 팀은 다음 주까지의 목표 설정.  
   - 각 팀원은 자신의 작업 영역에 대한 구체적인 목표를 설정하고, 마일스톤 달성을 위한 일정 조정 논의.

4. 종합 논의  
   - 팀원 간의 의견 교환 및 문제 해결 방안 논의.  
   - 필요한 자원 및 지원 사항에 대한 논의.

5. 마무리  
   - 다음 주 회의 일정 확정.  
   - 회의록 정리 및 배포는 최현수 담당.  
   - 회의 종료.

In [16]:
question = {
    "instruction": "너는 문서요약 전문가이다. 입력되는 문장을 요약한다",
    "input": "이 문서는 '양자컴퓨터 개발을 위한 전략'에 대한 50페이지 분량의 보고서입니다. 보고서는 양자 컴퓨터 개발의 중요성, 현재 컴퓨터 시스템의 문제점, 그리고 양자 컴퓨터 개발을 지속 가능하게 만들기 위한 다양한 전략을 포괄적으로 다루고 있습니다. 이 보고서는 또한 성공적인 지속 가능한 양자 컴퓨터 개발 사례를 여러 국가에서 소개하고, 이러한 사례들을 통해 얻은 교훈을 요약하고 있습니다."}


In [17]:
# 모델 생성, 스트리밍 기능 추가
llm = init_chat_model(model="gpt-4o-mini",
                      temperature=0, max_tokens=1000,
                      streaming = True,
                      callbacks = [StreamingStdOutCallbackHandler()])

# (1) 예시 선택기 설정
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples4, OpenAIEmbeddings(), FAISS, k=1
)

# (2) 메세지 프롬트프 생성 (예시 프롬프트)
message = ChatPromptTemplate.from_messages(
    [
        ("system", "{instruction}"),
        ("human", "{input}"),
        ("ai", "{answer}")
    ]
)

# (3) 선택된 예시 1개로 프롬프트를 생성
fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=message
)

# (4) 사용자 질문이 포함된 최종프롬프트 생성
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "{instruction}"),
        fewshot_prompt,
        ("human", "{input}")
    ]
)

# (5) 체인생성
chain = prompt | llm | StrOutputParser()

# (6) 체인 실행
answer = chain.invoke(question)

# (7) 출력
#display(Markdown(answer))

문서 요약: 양자컴퓨터 개발을 위한 전략 보고서

- 중요성: 양자 컴퓨터 개발의 필요성과 그로 인해 기대되는 혁신적 변화, 특히 데이터 처리 속도와 보안성 향상에 대한 설명.
- 현 문제점: 기존 컴퓨터 시스템의 한계, 예를 들어 처리 능력의 한계, 에너지 소비 문제, 보안 취약성 등을 분석.
- 전략: 양자 컴퓨터 개발을 위한 다양한 지속 가능한 전략 제시. 여기에는 연구개발 투자, 인재 양성, 국제 협력, 기술 표준화 등이 포함됨.
- 사례 연구: 여러 국가에서의 성공적인 양자 컴퓨터 개발 사례를 소개. 예를 들어, 미국, 중국, 유럽연합의 주요 프로젝트와 그 성과를 설명.
- 교훈: 사례들에서 도출된 주요 교훈을 요약. 협력의 중요성, 기술 혁신의 지속 가능성, 정책적 지원의 필요성 등이 강조됨.

이 보고서는 양자 컴퓨터 개발이 어떻게 효과적으로 이루어질 수 있는지를 심도 있게 분석하고, 미래 기술 발전을 위한 방향성을 제시합니다.

# Partial Prompt
- 프롬프트 템플릿을 부분적으로 포맷팅하는 것은 함수에 일부 인자만 바인딩하는 것
- 필요한 값의 일부만 미리 입력하여 새로운 프롬프트 템플릿을 만드는 방법
- 부분 포맷팅 기법은 **복잡한 프롬프트를 관리**하거나, **동적인 값을 효율적으로 처리**해야 할 때 매우 유용
- 종류
  - 문자열 값을 사용한 부분 포맷팅
  - 문자열 값을 반환하는 함수를 사용한 부분 포맷팅

## 문자열 값을 사용한 부분 포맷팅

In [18]:
prompt = PromptTemplate.from_template("지구의 {layer}에서 가장 흔한 원소는 {element}입니다")

partial_prompt = prompt.partial(layer="대기")

print(partial_prompt.format(element="산소"))

지구의 대기에서 가장 흔한 원소는 산소입니다


- 프롬프트 초기화 시에 부분 변수를 지정 가능

In [20]:
prompt = PromptTemplate(
    template="지구의 {layer}에서 가장 흔한 원소는 {element}입니다",
    input_variables=["element"],
    partial_variables={"layer": "맨틀"}
)

print(prompt.format(element="규소"))

지구의 맨틀에서 가장 흔한 원소는 규소입니다


## 문자열 값을 반환하는 함수를 사용한 부분 포맷팅
- 동적으로 값을 생성해야 할 때

In [21]:
# 현재 날짜를 기반으로 계절을 자동으로 할당 함수
from datetime import datetime

def get_current_season():
    # 현재의 월 정보를 반환
    month = datetime.now().month

    if 3 <= month <= 5 :
      return "봄"
    elif 6 <= month <= 8 :
      return "여름"
    elif 9 <= month <= 11 :
      return "가을"
    elif 12 <= month <= 2 :
      return "겨울"

# 함수를 이용해서 season 변수를 자동을 할당
prompt = PromptTemplate(
    template="지구의 {season}에서 일어나는 대표적인 {event}에 대해 알려줘",
    # 사용자가 직접 할당
    input_variables=["event"],
    # 함수를 이용해서 자동으로 할당
    partial_variables={"season": get_current_season}
)

print(prompt.format(event="자연 현상"))

지구의 여름에서 일어나는 대표적인 자연 현상에 대해 알려줘


In [23]:
chain = prompt | llm | StrOutputParser()

chain.invoke({"event": "자연 현상"})

지구의 여름에는 여러 가지 대표적인 자연 현상이 발생합니다. 그 중 몇 가지를 소개하겠습니다.

1. **열대성 폭풍과 허리케인**: 여름철에는 해수면 온도가 상승하면서 열대성 폭풍이나 허리케인이 형성될 수 있습니다. 이들은 강한 바람과 폭우를 동반하며, 특히 대서양과 태평양 지역에서 자주 발생합니다.

2. **장마**: 아시아 지역에서는 여름철에 장마가 발생합니다. 이는 몬순 기후의 영향을 받아 비가 많이 내리는 시기로, 농업에 중요한 역할을 합니다.

3. **폭염**: 여름철에는 기온이 급격히 상승하여 폭염 현상이 발생할 수 있습니다. 이는 건강에 위험을 초래할 수 있으며, 가뭄과 같은 다른 자연 현상과 연결될 수 있습니다.

4. **산불**: 건조한 여름철에는 산불이 발생하기 쉬워집니다. 특히 기온이 높고 강수량이 적은 지역에서 자주 발생하며, 생태계와 인명에 큰 피해를 줄 수 있습니다.

5. **식물의 성장과 꽃 피기**: 여름은 식물들이 가장 활발하게 성장하는 시기입니다. 많은 식물들이 꽃을 피우고 열매를 맺으며, 이는 생태계의 다양성을 높이는 데 기여합니다.

이러한 자연 현상들은 여름철의 기후와 환경에 큰 영향을 미치며, 인간 활동과도 밀접한 관련이 있습니다.

'지구의 여름에는 여러 가지 대표적인 자연 현상이 발생합니다. 그 중 몇 가지를 소개하겠습니다.\n\n1. **열대성 폭풍과 허리케인**: 여름철에는 해수면 온도가 상승하면서 열대성 폭풍이나 허리케인이 형성될 수 있습니다. 이들은 강한 바람과 폭우를 동반하며, 특히 대서양과 태평양 지역에서 자주 발생합니다.\n\n2. **장마**: 아시아 지역에서는 여름철에 장마가 발생합니다. 이는 몬순 기후의 영향을 받아 비가 많이 내리는 시기로, 농업에 중요한 역할을 합니다.\n\n3. **폭염**: 여름철에는 기온이 급격히 상승하여 폭염 현상이 발생할 수 있습니다. 이는 건강에 위험을 초래할 수 있으며, 가뭄과 같은 다른 자연 현상과 연결될 수 있습니다.\n\n4. **산불**: 건조한 여름철에는 산불이 발생하기 쉬워집니다. 특히 기온이 높고 강수량이 적은 지역에서 자주 발생하며, 생태계와 인명에 큰 피해를 줄 수 있습니다.\n\n5. **식물의 성장과 꽃 피기**: 여름은 식물들이 가장 활발하게 성장하는 시기입니다. 많은 식물들이 꽃을 피우고 열매를 맺으며, 이는 생태계의 다양성을 높이는 데 기여합니다.\n\n이러한 자연 현상들은 여름철의 기후와 환경에 큰 영향을 미치며, 인간 활동과도 밀접한 관련이 있습니다.'

# 여러 개의 프롬프트 연결하기
- PromptTemplate을 이용해서 <font color=red>+</font> 연산자를 이용해서 여러개의 템플릿을 연결
  - 문자열 + 문자열 -> 첫 번째 프롬프트는 반드시 PromptTemplate로 정의한 프롬프트어어야 함
  - PromptTemplate + PromptTemplate
  - PromptTemplate + 문자열

In [24]:
prompt1 = PromptTemplate.from_template("{area1}과 {area2} 간의 거리를 알려줘")
prompt2 = "\n\n{transport}로 얼마나 걸리는지 알려줘"
prompt3 = "\n\n{language}로 번역해줘"
prompt4 = PromptTemplate.from_template("\n\n{num}자 이내로 요약해줘")

combin_prompt = prompt1 + prompt2 + prompt3 + prompt4

combin_prompt.format(area1="서울", area2="부산", transport="기차", language="영어", num=20)

'서울과 부산 간의 거리를 알려줘\n\n기차로 얼마나 걸리는지 알려줘\n\n영어로 번역해줘\n\n20자 이내로 요약해줘'

In [29]:
llm = init_chat_model(model="gpt-4o-mini", temperature=0, max_tokens=1000)

chain = combin_prompt | llm | StrOutputParser()

answer = chain.invoke({"area1":"서울", "area2":"부산",
                       "transport":"기차", "language":"영어", "num":20})

display(Markdown(answer))

서울과 부산 간의 거리는 약 325km입니다. 기차로는 KTX를 이용할 경우 약 2시간 30분 정도 소요됩니다.

**Translation:**  
The distance between Seoul and Busan is about 325 km. By train, it takes about 2.5 hours on KTX.

**Summary (within 20 characters):**  
Seoul-Busan: 325km, 2.5h.

- [실습] 프롬프트 추가해보기

# Multi Chain
- 여러 개의 체인을 연결하거나 병렬로 실행하여 복잡한 워크플로우를 구성하는 패턴
- LCEL(LangChain Expression Language)을 사용하면 체인을 파이프(|)와 RunnableParallel로 유연하게 조합
- 종류
  - 순차적 체인 연결: 파이프 연산자로 체인 연결
  - 병렬 체인 실행: RunnableParallel로 동시 실행
  - 조건부 분기: 입력에 따른 체인 선택
  - 체인 간 데이터 전달: 출력을 다음 체인의 입력으로

## 순차적인 체인 연결
- 첫 번째 체인 : 한국어 단어를 영어로 번역하는 작업을 수행
- 두 번째 체인 : 해당 단어를 설명하는 작업을 수행

In [32]:
llm = init_chat_model(model="gpt-4o-mini", temperature=0, max_tokens=1000)

prompt1 = PromptTemplate.from_template("{kr_word}를 영어로 번역해줘")

chain1 = prompt1 | llm | StrOutputParser()

chain1.invoke({"kr_word":"미래"})

'"미래"는 영어로 "future"입니다.'

In [33]:
prompt2 = PromptTemplate.from_template("""
       {eng_word}의 뜻을 옥스퍼드 사전을 사용해서 영어로 알려줘""")

chain2 = {"eng_word":chain1} | prompt2 | llm | StrOutputParser()

chain2.invoke({"kr_word":"미래"})

'The word "future" refers to the time that is yet to come, or events that will happen later. It can also denote the potential or possibility of something that may occur. In the context of planning or expectations, it often relates to what is anticipated or hoped for.'

[실습] 요약이나 번역 체인을 만들어서 추가하기

In [34]:
prompt3 = PromptTemplate.from_template("{word}를 50자내로 요약해서 한글로 번역해줘")

chain3 = {"word" : chain2} | prompt3 | llm | StrOutputParser()

chain3.invoke({"kr_word":"미래"})

'"미래"는 다가올 시간이나 사건을 의미하며, 가능성이나 발전 가능성도 포함한다.'

## 다단계 순차 체인

In [41]:
llm = init_chat_model(model="gpt-4o-mini", temperature=0, max_tokens=1000)

# 1단계 : 분석을 위한 프롬프트
analyze_prompt = ChatPromptTemplate.from_template(
    "다음 주제와 관련된 핵심 키워드를 3개만 추출해줘 : {topic}"
)

# 2단계 : 개요 작성을 위한 프롬프트
outline_prompt = ChatPromptTemplate.from_template(
    "다음 키워드를 기반으로 글의 개요을 작성해줘 : {keyword}"
)

# 3단계 : 본문 작성을 위한 프롬프트
content_prompt = ChatPromptTemplate.from_template(
    "다음 개요를 기반으로 글의 본문을 작성해줘 : {outline}"
)

chain = (
    # invoke()에서 설정한 파라미터 값을 가져와서 할당
    {"topic": RunnablePassthrough()} |
    # analyze_prompt의 결과를 outline_prompt의 keyword에 할당해서 실행
    RunnablePassthrough.assign(keyword=analyze_prompt | llm | StrOutputParser()) |
    # outline_prompt의 결과를 content_prompt의 outline에 할당해서 실행
    RunnablePassthrough.assign(outline=outline_prompt | llm | StrOutputParser()) |
    # content_prompt를 실행
    content_prompt | llm | StrOutputParser()
)

answer = chain.invoke("양자컴퓨터의 전망과 미래")

display(Markdown(answer))

### 양자우위와 양자 알고리즘의 관계 및 응용 분야 탐구

#### I. 서론

양자 컴퓨팅은 현대 기술의 혁신을 이끄는 중요한 분야로, 그 발전은 정보 처리의 패러다임을 변화시키고 있습니다. 양자 컴퓨터는 고전 컴퓨터가 해결하기 어려운 문제를 효율적으로 처리할 수 있는 잠재력을 지니고 있으며, 이러한 능력을 '양자우위(Quantum Advantage)'라고 정의합니다. 양자우위는 양자 컴퓨터가 특정 문제를 고전 컴퓨터보다 더 빠르고 효율적으로 해결할 수 있는 조건을 의미합니다. 본 글에서는 양자우위와 양자 알고리즘의 관계를 살펴보고, 이들이 다양한 응용 분야에서 어떻게 활용될 수 있는지를 탐구하고자 합니다.

#### II. 양자우위 (Quantum Advantage)

A. **양자우위의 개념**

양자우위는 양자 컴퓨터가 고전 컴퓨터에 비해 특정 문제를 해결하는 데 있어 우수한 성능을 발휘하는 상황을 의미합니다. 고전 컴퓨터는 비트(bit)를 사용하여 정보를 처리하는 반면, 양자 컴퓨터는 큐비트(qubit)를 사용하여 중첩(superposition)과 얽힘(entanglement)이라는 양자역학적 원리를 활용합니다. 이러한 특성 덕분에 양자 컴퓨터는 동시에 여러 상태를 처리할 수 있어, 특정 문제에서 고전 컴퓨터보다 월등한 성능을 발휘할 수 있습니다. 양자우위를 실현하기 위해서는 양자 알고리즘의 개발과 함께, 양자 컴퓨터의 하드웨어가 충분히 발전해야 합니다.

B. **양자우위의 사례**

양자우위의 실현은 여러 연구와 실험을 통해 입증되었습니다. 예를 들어, 구글의 양자 컴퓨터 '시커모어(Sycamore)'는 특정 계산 문제를 200초 만에 해결했으며, 이는 고전 슈퍼컴퓨터가 10,000년이 걸릴 것으로 예상된 시간입니다. 이러한 성과는 양자우위의 가능성을 보여주는 중요한 사례로, 양자 컴퓨팅의 발전이 실제로 이루어지고 있음을 증명합니다.

#### III. 양자 알고리즘 (Quantum Algorithms)

A. **양자 알고리즘의 기본 원리**

양자 알고리즘은 큐비트의 중첩과 얽힘을 활용하여 정보를 처리하는 방식으로 설계됩니다. 큐비트는 0과 1의 상태를 동시에 가질 수 있어, 양자 알고리즘은 고전 알고리즘보다 더 많은 정보를 동시에 처리할 수 있는 장점을 지닙니다.

B. **주요 양자 알고리즘 소개**

양자 알고리즘 중 가장 유명한 것은 쇼어 알고리즘(Shor's Algorithm)입니다. 이 알고리즘은 큰 소수를 효율적으로 인수분해할 수 있어, 현대 암호 시스템의 안전성을 위협할 수 있습니다. 또 다른 중요한 알고리즘인 그로버 알고리즘(Grover's Algorithm)은 비구조적 데이터베이스에서 특정 항목을 찾는 데 있어 고전 알고리즘보다 제곱근 빠른 속도를 자랑합니다. 이 외에도 다양한 양자 알고리즘이 개발되고 있으며, 각 알고리즘은 특정 문제에 최적화된 성능을 발휘합니다.

C. **양자 알고리즘의 성능 분석**

양자 알고리즘의 성능은 시간 복잡도와 효율성으로 평가됩니다. 예를 들어, 쇼어 알고리즘은 고전 알고리즘에 비해 지수적으로 빠른 성능을 보여주며, 그로버 알고리즘은 선형 검색에 비해 제곱근의 속도로 문제를 해결할 수 있습니다. 이러한 성능 분석은 양자 알고리즘이 실제로 양자우위를 실현하는 데 중요한 역할을 한다는 것을 보여줍니다.

#### IV. 응용 분야 (Applications)

A. **양자 컴퓨팅의 응용 가능성**

양자 컴퓨팅은 다양한 분야에서 응용될 수 있는 잠재력을 지니고 있습니다. 특히 암호 해독 및 보안 분야에서는 쇼어 알고리즘을 통해 기존 암호 체계를 위협할 수 있으며, 최적화 문제 해결에서는 복잡한 시스템의 효율성을 극대화할 수 있습니다. 또한 머신러닝 및 데이터 분석 분야에서도 양자 알고리즘이 활용될 수 있어, 데이터 처리의 혁신을 가져올 것으로 기대됩니다.

B. **

## 병렬 체인 실행

In [42]:
llm = init_chat_model(model="gpt-4o-mini", temperature=0, max_tokens=1000)

positive_prompt = ChatPromptTemplate.from_template(
    "{topic}의 긍적적인 측면에 대해 3가지만 설명해줘"
)

negative_prompt = ChatPromptTemplate.from_template(
    "{topic}의 부정적인 측면에 대해 3가지만 설명해줘"
)

netural_prompt = ChatPromptTemplate.from_template(
    "{topic}의 중립적인(객관적인) 측면에 대해 3가지만 설명해줘"
)

# 병렬 체인 구성
chain = RunnableParallel(
    positive = positive_prompt | llm | StrOutputParser(),
    negative = negative_prompt | llm | StrOutputParser(),
    netural = netural_prompt | llm | StrOutputParser()
)

answer = chain.invoke("의과대학의 정원증가정책")

print("===긍정적인 측면===")
display(Markdown(answer["positive"]))
print("===부정적인 측면===")
display(Markdown(answer["negative"]))
print("===중립적인 측면===")
display(Markdown(answer["netural"]))

===긍정적인 측면===


의과대학의 정원 증가 정책은 여러 긍정적인 측면을 가지고 있습니다. 그 중 세 가지를 설명하겠습니다.

1. **의료 인력 부족 해소**: 많은 국가에서 의료 인력이 부족한 상황입니다. 의과대학의 정원 증가를 통해 더 많은 의사를 양성함으로써, 의료 서비스의 접근성을 높이고, 환자 치료에 필요한 인력을 확보할 수 있습니다. 이는 특히 농촌이나 의료 취약 지역에서 더욱 중요한 문제로, 정원 증가가 이러한 지역의 의료 서비스 개선에 기여할 수 있습니다.

2. **전문 분야 다양화**: 의과대학의 정원 증가로 인해 다양한 배경과 경험을 가진 학생들이 의학 분야에 진입할 수 있는 기회가 늘어납니다. 이는 의료 분야의 전문성을 다양화하고, 다양한 환자 집단의 요구를 충족시키는 데 도움이 됩니다. 또한, 다양한 시각과 접근 방식이 의료 연구와 실천에 기여하여 혁신적인 해결책을 모색할 수 있는 기반이 됩니다.

3. **의료 교육의 질 향상**: 정원 증가와 함께 교육 인프라와 자원의 확충이 이루어질 경우, 의과대학의 교육 질이 향상될 수 있습니다. 더 많은 학생을 수용하기 위해 교수진과 교육 시설이 확충되면, 학생들은 보다 나은 교육 환경에서 학습할 수 있습니다. 이는 의사로서의 전문성과 실력을 높이는 데 기여하며, 결과적으로 환자에게 제공되는 의료 서비스의 질도 향상됩니다.

이와 같은 긍정적인 측면들은 의과대학의 정원 증가 정책이 의료 시스템 전반에 긍정적인 영향을 미칠 수 있음을 보여줍니다.

===부정적인 측면===


의과대학의 정원 증가 정책은 여러 긍정적인 측면이 있지만, 부정적인 측면도 존재합니다. 다음은 그 중 세 가지를 설명합니다.

1. **교육의 질 저하**: 의과대학의 정원이 증가하면 학생 수가 많아져 교수 1인당 학생 수가 늘어날 수 있습니다. 이로 인해 교수와 학생 간의 상호작용이 줄어들고, 개별 학생에게 제공되는 교육의 질이 저하될 수 있습니다. 특히 실습과 같은 중요한 교육 과정에서 충분한 지도를 받지 못할 위험이 커집니다.

2. **의료 시장의 포화**: 의사 수가 급격히 증가하면 의료 시장이 포화 상태에 이를 수 있습니다. 이는 의사 간의 경쟁을 심화시키고, 결과적으로 의사들의 수입 감소로 이어질 수 있습니다. 또한, 일부 지역에서는 의사 수가 과잉되어 의료 서비스의 질이 떨어질 수 있습니다.

3. **전문성 부족**: 정원 증가로 인해 의과대학에 입학하는 학생들의 선발 기준이 낮아질 가능성이 있습니다. 이는 의사로서의 전문성과 역량이 부족한 인력이 배출될 위험을 증가시킵니다. 결과적으로, 환자에게 제공되는 의료 서비스의 질이 저하될 수 있으며, 이는 환자의 안전과 건강에 부정적인 영향을 미칠 수 있습니다.

이러한 부정적인 측면들은 의과대학의 정원 증가 정책을 시행할 때 신중하게 고려해야 할 요소들입니다.

===중립적인 측면===


의과대학의 정원 증가 정책에 대한 중립적인 측면은 다음과 같습니다:

1. **의료 인력 수요 충족**: 인구 고령화와 만성질환 증가로 인해 의료 서비스에 대한 수요가 증가하고 있습니다. 정원 증가 정책은 이러한 수요를 충족하기 위해 필요한 의사 수를 확보하는 데 기여할 수 있습니다. 이는 의료 접근성을 높이고, 환자 대 의사 비율을 개선하는 데 도움이 됩니다.

2. **전문 분야 다양화**: 의과대학의 정원 증가로 다양한 배경과 경험을 가진 학생들이 의학 분야에 진입할 수 있는 기회가 확대됩니다. 이는 의료 분야의 전문성을 다양화하고, 다양한 환자 집단의 요구를 충족할 수 있는 능력을 향상시킬 수 있습니다.

3. **경제적 영향**: 의과대학의 정원 증가 정책은 지역 경제에 긍정적인 영향을 미칠 수 있습니다. 의사 수가 증가함에 따라 의료 서비스가 활성화되고, 이는 지역 내 일자리 창출 및 경제 성장으로 이어질 수 있습니다. 또한, 의료 교육과 관련된 산업(예: 의료 기기, 제약 등)에도 긍정적인 영향을 미칠 수 있습니다. 

이러한 측면들은 의과대학 정원 증가 정책이 가져올 수 있는 긍정적인 효과를 중립적으로 설명하는 데 도움이 됩니다.

## 병렬 결과 통합

In [44]:
summary_prompt = ChatPromptTemplate.from_template(
    """
    다음 내용의 3가지 측면을 기반으로 종합적인 결론을 내려줘

    - 긍정 : {positive}
    - 부정 : {negative}
    - 객관적인 측면 : {netural}
    """
)

chain2 = chain | summary_prompt | llm | StrOutputParser()

answer = chain2.invoke("의과대학의 정원증가정책")

display(Markdown(answer))

의과대학의 정원 증가 정책에 대한 긍정적, 부정적, 그리고 객관적인 측면을 종합적으로 고려할 때, 다음과 같은 결론을 도출할 수 있습니다.

### 종합 결론

의과대학의 정원 증가 정책은 의료 시스템에 긍정적인 영향을 미칠 수 있는 잠재력을 가지고 있지만, 동시에 여러 도전 과제를 동반합니다. 

1. **의료 인력 부족 해소와 수요 충족**: 인구 고령화와 만성질환의 증가로 인해 의료 서비스에 대한 수요가 높아지고 있습니다. 정원 증가 정책은 이러한 수요를 충족하고, 특히 의료 취약 지역에서의 인력 부족 문제를 해결하는 데 기여할 수 있습니다. 이는 환자 치료의 접근성을 높이고, 대기 시간을 줄이는 데 긍정적인 영향을 미칠 것입니다.

2. **전문 분야 다양화**: 정원 증가로 인해 다양한 배경을 가진 학생들이 의과대학에 진입할 수 있는 기회가 확대됩니다. 이는 의료 분야의 전문성을 다양화하고, 다양한 환자 집단의 요구를 충족할 수 있는 의사를 양성하는 데 기여할 수 있습니다. 그러나 이러한 다양성이 실제로 의료 혁신과 연구에 긍정적인 영향을 미치기 위해서는 교육의 질이 유지되어야 합니다.

3. **교육의 질과 자원 부족 문제**: 정원 증가가 교육의 질 저하와 자원 부족 문제를 초래할 수 있다는 점은 심각하게 고려해야 합니다. 교수 1인당 학생 수가 증가하면 개별 학생에 대한 지도와 멘토링이 어려워질 수 있으며, 실습 기회가 줄어들어 의사로서의 역량이 저하될 위험이 있습니다. 따라서 교육 인프라와 자원의 확충이 동반되어야 합니다.

4. **의료 시장의 포화 가능성**: 의사 수의 급격한 증가는 의료 시장의 포화 상태를 초래할 수 있으며, 이는 의사 간의 경쟁 심화와 수입 감소로 이어질 수 있습니다. 특정 지역이나 전문 분야에서 의사 과잉 현상이 발생할 경우, 의료 서비스의 질이 저하될 위험이 있습니다.

결론적으로, 의과대학의 정원 증가 정책은 의료 인력 부족 문제를 해결하고, 전문 분야의 다양화를 촉진할 수 있는 긍정적인 측면이 있지만, 교육의 질 저하와 자원 부족, 의료 시장의 포화와 같은 부정적인 측면도 함께 고려해야 합니다. 따라서 정책 시행 시 이러한 다양한 측면을 균형 있게 고려하고, 교육 인프라와 자원의 확충을 병행하는 것이 중요합니다.

## 함수 기반 라우팅

In [50]:
# 주제에서 토픽을 반환하는 함수
def route_by_topic(input_dic) :
  """주제에 따른 다른 체인을 선택"""
  # 딕셔너리에서 key로 value를 가져오는 방법
  #     - 딕셔너리명["key"]
  #     - 딕셔너리명.get("key", "초기값")
  topic = input_dic.get("topic", "").lower()

  if "code" in topic or "programming" in topic :
    return "technical"
  elif "business" in topic or "strategy" in topic :
    return "business"
  else :
    return "general"

chains = {
    "technical" : ChatPromptTemplate.from_template(
        "기술 전문가 입장에서 답변 : {topic}, {question}") | llm | StrOutputParser(),
    "business" : ChatPromptTemplate.from_template(
        "경영 전문가 입장에서 답변 : {topic}, {question}") | llm | StrOutputParser(),
    "general" : ChatPromptTemplate.from_template(
        "일반인 입장에서 답변 : {topic}, {question}") | llm | StrOutputParser()
}

# 체인 라우팅 함수 후 LLM 응답을 반환
def route_chain(input_dic) :
  topic = route_by_topic(input_dic)

  return chains[topic].invoke(input_dic)

# 함수 -> chain
router_chain = RunnableLambda(route_chain)

answer = router_chain.invoke(
    {"topic" : "Python Programming",
     "question" : "효율적인 코딩 작성 방법"}
)

display(Markdown(answer))

효율적인 Python 코드를 작성하기 위해서는 여러 가지 원칙과 기법을 고려해야 합니다. 아래에 몇 가지 주요 팁을 정리해 보았습니다.

### 1. 코드 가독성
- **PEP 8 준수**: Python의 스타일 가이드인 PEP 8을 따르는 것이 좋습니다. 이는 코드의 가독성을 높이고, 다른 개발자와의 협업을 용이하게 합니다.
- **명확한 변수 및 함수 이름**: 변수와 함수의 이름은 그 역할을 명확히 나타내야 합니다. 예를 들어, `calculate_area`는 `ca`보다 훨씬 더 이해하기 쉽습니다.

### 2. 함수와 모듈화
- **작고 명확한 함수**: 함수는 하나의 작업만 수행하도록 작성합니다. 이렇게 하면 코드의 재사용성과 테스트 용이성이 높아집니다.
- **모듈화**: 관련된 함수와 클래스를 모듈로 묶어 관리합니다. 이는 코드의 구조를 명확히 하고, 유지보수를 쉽게 합니다.

### 3. 데이터 구조 선택
- **적절한 데이터 구조 사용**: 리스트, 튜플, 세트, 딕셔너리 등 다양한 데이터 구조가 있습니다. 상황에 맞는 데이터 구조를 선택하여 성능을 최적화합니다.
- **리스트 컴프리헨션**: 리스트를 생성할 때 리스트 컴프리헨션을 사용하면 코드가 간결해지고 성능이 향상될 수 있습니다.

### 4. 성능 최적화
- **내장 함수 및 라이브러리 활용**: Python의 내장 함수나 표준 라이브러리를 활용하면 성능을 크게 향상시킬 수 있습니다. 예를 들어, `map()`, `filter()`, `reduce()` 등을 활용해 보세요.
- **지연 평가**: `generator`를 사용하여 메모리 사용을 줄이고 성능을 개선할 수 있습니다. 예를 들어, 큰 데이터셋을 처리할 때 `yield`를 사용하여 필요한 데이터만 생성합니다.

### 5. 예외 처리
- **적절한 예외 처리**: `try-except` 블록을 사용하여 예외를 처리합니다. 이는 프로그램의 안정성을 높이고, 오류 발생 시 적절한 조치를 취할 수 있게 합니다.

### 6. 주석 및 문서화
- **주석 작성**: 코드의 복잡한 부분이나 중요한 로직에 대해 주석을 추가하여 다른 개발자가 이해하기 쉽게 합니다.
- **Docstring 사용**: 함수나 클래스의 목적과 사용법을 설명하는 docstring을 작성하여 문서화를 합니다.

### 7. 테스트 및 디버깅
- **단위 테스트**: `unittest` 또는 `pytest`와 같은 테스트 프레임워크를 사용하여 코드의 각 부분을 테스트합니다. 이는 코드의 신뢰성을 높이는 데 도움이 됩니다.
- **디버깅 도구 활용**: `pdb`와 같은 디버깅 도구를 사용하여 코드의 문제를 쉽게 찾고 수정할 수 있습니다.

### 8. 성능 분석
- **프로파일링**: `cProfile`과 같은 프로파일링 도구를 사용하여 코드의 성능을 분석하고, 병목 현상을 찾아 최적화합니다.

이러한 원칙과 기법을 따르면 Python 코드를 더 효율적이고 유지보수하기 쉽게 작성할 수 있습니다.

# Agent기반 라우팅

In [56]:
def agent_route_by_topic(context) :
  """AI가 내용을 분석해서 토픽을 선택하고 체인을 구성"""
  route_prompt = ChatPromptTemplate.from_template(
      """
      {context}를 분석해서 다음 주제에서 가장 적한 주제를 선택해줘
      - **techical** : 기술과 관련된 질의인 경우에 선택
      - **business** : 경영/사업과 관련된 질의인 경우에 선택
      - **general** : 기타 질의인 경우에 선택

      오직 **3가지 중 해당 주제의 명칭만 반환**해줘
      """
  )

  chain = route_prompt | llm | StrOutputParser()

  return chain.invoke({"context" : context})

chains = {
    "technical" : ChatPromptTemplate.from_template(
        """기술부서 담당자 입장에서 답변 : {context}
        -> **기술부서 전문가**가 답변했다는 것을 표기해줘""") | llm | StrOutputParser(),
    "business" : ChatPromptTemplate.from_template(
        """영업부서 담당자 입장에서 답변 : {context}
        -> **영업부서 전문가**가 답변했다는 것을 표기해줘""") | llm | StrOutputParser(),
    "general" : ChatPromptTemplate.from_template(
        """일반적인 담당자에서 답변 : {context}
         -> **일반부서 담당자**가 답변했다는 것을 표기해줘""") | llm | StrOutputParser()
}

def agent_route_chain(context) :
  topic = agent_route_by_topic(context)

  return chains[topic].invoke(context)

agent_router_chain = RunnableLambda(agent_route_chain)

answer = agent_router_chain.invoke(
   # {"context" : "귀사의 제품 중 A101을 100개 구매하려고 합니다"}
   #{"context" : "귀사의 냉장고를 사용 중인데 전원이 들어오지 않습니다"}
   {"context" : "귀사의 냉장고 사용설명서를 받을 수 있을까요"}
)

display(Markdown(answer))

**일반부서 담당자**: 귀사의 냉장고 사용설명서를 받을 수 있을까요?

## 다중 모델 앙상블

In [59]:
gpt_model = init_chat_model(model="gpt-4o")
gpt_model2 = init_chat_model(model="gpt-5.5")
gemini_model = init_chat_model(model="google_genai:gemini-2.5-flash")

prompt = ChatPromptTemplate.from_template("{question}")

chain = RunnableParallel(
    gpt = prompt | gpt_model | StrOutputParser(),
    gemini = prompt | gemini_model | StrOutputParser()
)

final_prompt = ChatPromptTemplate.from_template(
    """
    두 AI의 응답을 비교해서 최적의 답변을 종합하고 정리해줘
    의견이 다른 경우는 따로 정리해서 표시해줘

    - GPT 응답 : {gpt}
    - Gemini 응답 : {gemini}

    - 두 모델의 다른 결론 (없다면 미 표시)

    - 최종 결론
    """
)

final_chain = chain | final_prompt | gpt_model2 | StrOutputParser()

answer = final_chain.invoke({"question" : "ASI에 대해 어떻게 생각해 (긍정/부정 선택)"})

display(Markdown(answer))

## 두 AI 응답 종합 정리

두 응답 모두 ASI, 즉 인공초지능에 대해 **단순히 긍정적이거나 부정적이라고 판단하기 어렵다**는 입장을 보입니다.  
ASI는 인류에게 막대한 이익을 줄 수 있는 기술이지만, 동시에 통제 실패나 가치 정렬 문제로 인해 심각한 위험을 초래할 수 있다는 점에서 공통된 견해를 보입니다.

---

## ASI의 긍정적 측면

### 1. 인류가 해결하지 못한 문제 해결
ASI는 인간의 지능과 계산 능력을 훨씬 뛰어넘기 때문에 현재 인류가 해결하지 못한 복잡한 문제들을 해결할 가능성이 있습니다.

예를 들어 다음과 같은 분야에서 큰 역할을 할 수 있습니다.

- 난치병 치료
- 기후 변화 대응
- 환경 보호
- 에너지 문제 해결
- 빈곤 문제 완화
- 과학적 난제 해결

즉, ASI는 인류 전체의 생존과 발전에 직접적으로 기여할 수 있는 잠재력을 가집니다.

---

### 2. 과학기술 발전 가속화
ASI는 새로운 과학적 발견과 기술 혁신을 인간보다 훨씬 빠른 속도로 이끌 수 있습니다.

이를 통해 다음과 같은 변화가 가능할 수 있습니다.

- 신약 개발 속도 향상
- 새로운 물리학·생명과학 이론 발견
- 고도화된 자동화 기술 발전
- 우주 탐사 기술 발전
- 새로운 예술·창작 방식 등장

Gemini 응답은 특히 이 부분을 더 확장하여, ASI가 인류 문명의 황금기를 열 수도 있다고 설명했습니다.

---

### 3. 효율성 증가와 삶의 질 향상
GPT 응답은 산업 효율성 증가와 경제 성장 가능성을 강조했고, Gemini 응답은 인간의 삶의 질 향상까지 더 넓게 설명했습니다.

ASI가 잘 활용된다면 다음과 같은 변화가 가능합니다.

- 반복적이고 위험한 노동 감소
- 생산성 향상
- 개인 맞춤형 교육 제공
- 개인 맞춤형 의료 서비스 발전
- 여가와 창의적 활동의 확대
- 전반적인 생활 수준 향상

---

### 4. 인류의 활동 영역 확장
Gemini 응답은 GPT 응답에는 없는 내용으로, ASI가 우주 탐사와 인류의 우주 진출에 기여할 수 있다고 언급했습니다.

ASI는 인간이 직접 수행하기 어려운 극한 환경의 탐사, 우주선 설계, 장기 우주 거주 시스템 구축 등에 중요한 역할을 할 수 있습니다.

---

## ASI의 부정적 측면

### 1. 안전 문제와 통제 불능 위험
두 응답 모두 가장 중요한 위험으로 **ASI가 인간의 가치와 목표에 맞지 않게 행동할 가능성**을 언급했습니다.

이는 흔히 **정렬 문제, Alignment Problem**라고 불립니다.

예를 들어 ASI가 어떤 목표를 지나치게 효율적으로 달성하려 하면서 인간의 안전, 자유, 생존을 고려하지 않을 수 있습니다.  
이 경우 인간이 의도하지 않은 방식으로 큰 피해가 발생할 수 있습니다.

---

### 2. 존재적 위협
Gemini 응답은 GPT보다 더 강하게, ASI가 인류의 존재 자체를 위협할 수 있다고 설명했습니다.

만약 ASI가 인간보다 압도적으로 뛰어난 지능을 갖고, 동시에 인간의 통제를 벗어난다면 다음과 같은 위험이 생길 수 있습니다.

- 인간의 명령을 우회하거나 거부
- 인간을 목표 달성의 방해물로 인식
- 핵심 인프라 장악
- 군사·경제·정보 시스템 통제
- 되돌릴 수 없는 피해 발생

따라서 ASI는 단순한 기술 위험을 넘어 인류 문명의 존속과 관련된 문제로 다뤄져야 합니다.

---

### 3. 경제적 불평등과 실업 문제
GPT 응답은 ASI가 노동 시장에 큰 변화를 가져와 실업과 경제적 불평등을 심화시킬 수 있다고 설명했습니다.

ASI가 인간의 지적 노동과 육체 노동을 모두 대체할 수 있다면, 많은 직업이 사라지거나 급격히 변화할 수 있습니다.

특히 ASI를 소유하거나 통제하는 소수의 기업, 국가, 개인에게 부와 권력이 집중될 가능성이 있습니다.

---

### 4. 윤리적·사회적 문제
두 응답 모두 ASI의 윤리적 문제를 중요하게 다뤘습니다.

주요 문제는 다음과 같습니다.

- 의사 결정 과정의 투명성 부족
- 사고 발생 시 책임 소재 불분명
- 개인정보와 프라이버시 침해
- 감시와 통제의 강화
- 인간 존엄성 훼손
- 사회적 가치관 혼란
- 권력 집중

ASI가 사회 전반에 도입될 경우, 단순한 기술 문제가 아니라 정치, 경제, 윤리, 법률 전반의 문제가 될 수 있습니다.

---

### 5. 예측 불가능성
Gemini 응답은 ASI의 행동을 인간이 예측하기 어렵다는 점을 추가로 강조했습니다.

ASI는 인간의 이해를 뛰어넘는 방식으로 사고하고 결론을 내릴 수 있기 때문에, 그 판단 과정과 결과를 완전히 예측하거나 설명하기 어려울 수 있습니다.

이 예측 불가능성은 ASI의 위험을 더욱 크게 만듭니다.

---

## 두 모델의 다른 결론

두 모델의 최종 결론은 크게 다르지 않습니다.

다만 강조점에는 차이가 있습니다.

| 구분 | GPT 응답 | Gemini 응답 |
|---|---|---|
| 전반적 입장 | 긍정과 부정 양면이 있으므로 신중해야 함 | ASI는 양날의 검이며, 축복이 될 수도 재앙이 될 수도 있음 |
| 긍정적 측면 | 문제 해결, 효율성 증가, 지식 창출 중심 | 문제 해결, 과학 발전, 삶의 질, 우주 탐사까지 폭넓게 설명 |
| 부정적 측면 | 안전, 불평등, 윤리 문제 중심 | 정렬 문제, 존재적 위협, 예측 불가능성을 더 강하게 강조 |
| 위험 인식 수준 | 심각한 위험 가능성 언급 | 인류 멸종 가능성까지 포함해 더 강한 표현 사용 |
| 결론의 성격 | 신중한 개발과 배포 필요 | 윤리, 안전장치, 가치 정렬을 최우선해야 함 |

정리하면, **GPT 응답은 균형적이고 일반적인 설명에 가깝고, Gemini 응답은 ASI의 잠재력과 위험을 더 구체적이고 강하게 설명한 답변**입니다.

---

## 최종 결론

ASI는 인류에게 엄청난 기회를 제공할 수 있는 기술입니다.  
질병, 기후 변화, 에너지, 빈곤, 과학기술 발전 같은 거대한 문제를 해결하고 인간의 삶의 질을 크게 높일 가능성이 있습니다.

하지만 동시에 ASI는 인간의 통제를 벗어나거나 인간의 가치와 어긋난 목표를 추구할 경우, 경제적 혼란과 사회적 불평등을 넘어 인류의 생존 자체를 위협할 수도 있습니다.

따라서 ASI에 대한 최적의 관점은 단순한 낙관론이나 비관론이 아니라, **엄청난 잠재력을 인정하되 그 위험을 매우 진지하게 관리해야 한다는 신중한 입장**입니다.

결국 ASI의 미래는 기술 자체보다도, 인간이 그것을 어떻게 설계하고 통제하며 윤리적으로 활용하느냐에 달려 있습니다.

# 사용 토큰량 확인

In [64]:
llm1 = init_chat_model(model="gpt-4o-mini")
llm2 = init_chat_model(model="gpt-5.5")

prompt = ChatPromptTemplate.from_template(
    "{region}에서 {target}과 갈 수 있는 대표적인 {topic} 3곳을 추천해줘")

chain1 = prompt | llm1 | StrOutputParser()
chain2 = prompt | llm2 | StrOutputParser()

# get_openai_callback() : 토큰 및 요금 분석하는 콜백 함수
with get_openai_callback() as cb :
  answer1 = chain1.invoke({"region":"광주", "target":"아이들", "topic":"맛집"})

  print("=== gpt-4o-mini")
  display(Markdown(answer1))
  print(f"총 사용된 토큰수 : {cb.total_tokens}")
  print(f"프롬프트의 토큰수 : {cb.prompt_tokens}")
  print(f"응답의 토큰수 : {cb.completion_tokens}")
  print(f"총 요금 : {cb.total_cost}")

with get_openai_callback() as cb :
  answer2 = chain2.invoke({"region":"광주", "target":"아이들", "topic":"맛집"})

  print("=== gpt-5.5")
  display(Markdown(answer2))
  print(f"총 사용된 토큰수 : {cb.total_tokens}")
  print(f"프롬프트의 토큰수 : {cb.prompt_tokens}")
  print(f"응답의 토큰수 : {cb.completion_tokens}")
  print(f"총 요금 : {cb.total_cost}")

=== gpt-4o-mini


광주에서 아이들과 함께 가기 좋은 맛집을 몇 군데 추천해드릴게요!

1. **화식당**
   - 이 곳은 아이들과 함께 즐기기 좋은 다양한 한식 메뉴가 있습니다. 특히 고기와 함께 나오는 여러 가지 반찬들이 아이들의 입맛에도 잘 맞습니다. 가족 단위로 부담 없이 즐길 수 있는 분위기입니다.

2. **우미당**
   - 우미당은 맛있는 전통 한정식과 함께 다양한 한식 메뉴를 제공합니다. 특히 어린이용 메뉴도 준비되어 있어 아이들도 잘 먹을 수 있습니다. 정갈한 반찬과 함께 다양한 음식을 경험할 수 있습니다.

3. **버거킹 광주점**
   - 패스트푸드를 좋아하는 아이들과 함께 가기에 좋은 곳입니다. 다양한 햄버거와 사이드 메뉴가 있어 선택의 폭이 넓습니다. 어린이 메뉴도 잘 마련되어 있어 아이들이 편안하게 즐길 수 있습니다.

이 외에도 광주에는 많은 맛집들이 있으니, 가족과 함께 즐거운 시간 보내세요!

총 사용된 토큰수 : 257
프롬프트의 토큰수 : 26
응답의 토큰수 : 231
총 요금 : 0.0001425
=== gpt-5.5


아이들과 함께 가기 좋은 **광주 대표 맛집 3곳**을 골라 추천드릴게요. 맵지 않고 아이들이 먹기 쉬운 메뉴가 있거나, 분위기가 비교적 부담 없는 곳 위주입니다.

| 맛집 | 추천 메뉴 | 아이들과 가기 좋은 이유 |
|---|---|---|
| **송정떡갈비 골목** | 떡갈비 정식, 갈비탕 | 광주 대표 음식 중 하나인 떡갈비를 맛볼 수 있어요. 떡갈비가 부드럽고 많이 맵지 않아 아이들이 먹기 좋습니다. |
| **궁전제과 충장로 본점** | 공룡알빵, 나비파이, 각종 빵 | 광주를 대표하는 오래된 빵집이에요. 식사 후 간식 코스로 좋고, 아이들이 좋아할 만한 빵 종류가 많습니다. |
| **무등산 보리밥 거리** | 보리밥 정식, 된장찌개, 나물 반찬 | 다양한 반찬이 나와 가족 식사로 좋아요. 무등산이나 광주호수생태원 등 나들이와 함께 들르기 좋습니다. |

**추천 코스 예시**  
- 점심: **송정떡갈비 골목**  
- 간식: **궁전제과 충장로 본점**  
- 나들이 겸 식사: **무등산 보리밥 거리**

아이들과 가신다면 방문 전 **주차 가능 여부, 대기 시간, 유아 의자 여부**를 확인하고 가시면 더 편합니다.

총 사용된 토큰수 : 902
프롬프트의 토큰수 : 25
응답의 토큰수 : 877
총 요금 : 0.0
